# Preflight refits — SDL 2026 (single fitting session)

**Run 2026-07-16.** Every check modifies exactly one element of the layer4b army specification
(`layer4b_longwindow.py`: WV2-baseline ordered logit; same sample logic, controls, coding).
The replication gate below must pass before any check counts. New files only: this notebook,
`results/preflight_refits_table.csv`, `results/placebo_models_full.csv`.

## Verdicts

1. **Weights** — MERGE VALID (positional on `(wave, row_in_wave)`; 100% row-for-row agreement on 3 validation variables × 6 waves). Weighted `nr x WV4` = **+0.776** (sandwich SE 0.183, p = 2.3e-05, n = 8,915) vs +0.716 unweighted → **SURVIVES** (+8%, significant; institution contrast intact: police +0.474*, courts n.s.).
2. **Placebos** — refit **reproduces every stored value**: 5/5 required anchors MATCH at 3 dp, and all 15 stored interaction cells + 18/18 stored totals match; full coef/SE/z/p/n exported to `results/placebo_models_full.csv` (bases `full` and `common_8777`). No mismatches.
3. **rel_attendance control** — `nr x WV4` without it = **+0.726** (SE 0.181, p = 6.3e-05, n = 9,116) vs +0.716 → unchanged; police (+0.481\*\*) / courts (n.s.) contrast unchanged.
4. **left_right control** — `nr x WV4` without it = **+0.681** (SE 0.176, p = 1.1e-04) on n = **9,652** vs +0.716 on 8,915. Decomposition: control removal −0.059 (→ +0.657 on the same 8,915 rows), added respondents +0.024; all variants remain significant.
5. **Link function** — proportional odds **not rejected** on `nr` / `nr x WV4` (conservative pairwise cutpoint contrasts, all p > 0.24; point estimates decline monotonically toward higher cutpoints — reported, not significant). Binary high-confidence logit `nr x WV4` = **+1.323** (SE 0.322, p = 4.0e-05) → **consistent with the ordinal result: YES**.


In [1]:
# Setup, shared layer4b spec, REPLICATION GATE
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.tools.numdiff import approx_fprime, approx_hess
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
RESULTS = "results"
WAVES = ["WV2", "WV3", "WV4", "WV5", "WV6", "WV7"]
LATER = WAVES[1:]

# data prep — mirrors layer4b_longwindow.py lines 34-53 exactly
df = pd.read_csv(f"{RESULTS}/pooled_extended_v2.csv")
df = df.dropna(subset=["cluster"]).copy()
df["not_religious_dummy"] = df["rel_person"].map(
    lambda v: 1 if v in (2, 3) else (0 if v == 1 else np.nan))
for w in LATER:
    df[f"wave_{w}"] = (df["wave"] == w).astype(int)
    df[f"nr_x_{w}"] = df["not_religious_dummy"] * df[f"wave_{w}"]

PREDICTORS = [
    "not_religious_dummy", "rel_attendance", "rel_importance_god", "left_right",
    "age", "female", "education_raw", "income_raw", "married_dummy",
] + [f"wave_{w}" for w in LATER] + [f"nr_x_{w}" for w in LATER]
NR_TERMS = ["not_religious_dummy"] + [f"nr_x_{w}" for w in LATER]


def fit_ord(data, outcome, predictors, start=None):
    """layer4b fit: listwise dropna on outcome+predictors, drop constant columns, BFGS."""
    sub = data[[outcome] + predictors].dropna().copy()
    sub[outcome] = sub[outcome].astype(int)
    keep = [p for p in predictors if sub[p].nunique() > 1]
    res = OrderedModel(sub[outcome], sub[keep], distr="logit").fit(
        method="bfgs", maxiter=400, disp=False, start_params=start)
    assert res.mle_retvals.get("converged", False), f"non-convergence: {outcome}"
    return res, sub, keep


def wave_total(res, wave):
    """Total nr effect (main + interaction) with delta-method SE from the model
    covariance — same approach as layer4b_longwindow.py wave_total()."""
    main = "not_religious_dummy"
    if wave == "WV2":
        c, se = float(res.params[main]), float(res.bse[main])
    else:
        inter = f"nr_x_{wave}"
        if inter not in res.params.index:
            return np.nan, np.nan, np.nan
        L = np.zeros(len(res.params))
        L[res.params.index.get_loc(main)] = 1
        L[res.params.index.get_loc(inter)] = 1
        c = float(L @ res.params)
        se = float(np.sqrt(L @ res.cov_params() @ L))
    return c, se, 2 * (1 - stats.norm.cdf(abs(c / se)))


ROWS, SUMMARY = [], {}  # master tidy table + final summary registry

def add_row(check, inst, wave, term, coef, se, n, notes=""):
    z = coef / se if (se is not None and not np.isnan(se)) else np.nan
    p = 2 * (1 - stats.norm.cdf(abs(z))) if not np.isnan(z) else np.nan
    ROWS.append({"check": check, "institution": inst, "wave": wave, "term": term,
                 "coef": coef, "se": se, "z": z, "p": p, "n": n, "notes": notes})

def wave_of(term):
    return "" if term == "not_religious_dummy" else term.replace("nr_x_", "")

# ---------------- REPLICATION GATE ----------------
res_base, sub_base, keep_base = fit_ord(df, "conf_military", PREDICTORS)
pub = pd.read_csv(f"{RESULTS}/layer4b_longwindow_interactions.csv")
prow = pub.query("institution=='conf_military' and predictor=='nr_x_WV4'").iloc[0]
g_c, g_s = float(res_base.params["nr_x_WV4"]), float(res_base.bse["nr_x_WV4"])
g_p, g_n = float(res_base.pvalues["nr_x_WV4"]), len(sub_base)
print(f"refit:     nr_x_WV4 = {g_c:+.6f}  SE {g_s:.6f}  p {g_p:.4g}  n {g_n}")
print(f"published: nr_x_WV4 = {float(prow['coef']):+.6f}  SE {float(prow['se']):.6f}  "
      f"p {float(prow['p_value']):.4g}  n {int(prow['n_obs'])}")
assert abs(g_c - prow["coef"]) < 1e-4, "GATE FAILED on coef — STOP"
assert abs(g_s - prow["se"]) < 1e-4, "GATE FAILED on SE — STOP"
assert abs(g_p - prow["p_value"]) / prow["p_value"] < 1e-3, "GATE FAILED on p — STOP"
assert g_n == int(prow["n_obs"]) == 8915, "GATE FAILED on n — STOP"
print("REPLICATION GATE PASSED (coef/SE to 4+ decimals, p to <0.1%, n exact)")
for t in NR_TERMS:
    add_row("0_gate_baseline", "conf_military", wave_of(t), t,
            float(res_base.params[t]), float(res_base.bse[t]), g_n,
            "layer4b baseline refit; equals published layer4b_longwindow_interactions.csv")
SUMMARY["0_baseline"] = ("layer4b baseline (gate refit = published)", g_c, g_s, g_p, g_n, "")


refit:     nr_x_WV4 = +0.716155  SE 0.182816  p 8.953e-05  n 8915
published: nr_x_WV4 = +0.716155  SE 0.182816  p 8.953e-05  n 8915
REPLICATION GATE PASSED (coef/SE to 4+ decimals, p to <0.1%, n exact)


## Check 1 — survey weights

**1a feasibility.** `pool_final.py` `load_wave()` reads each source workbook with
`pd.read_excel(usecols=...)`, renames, and `pd.concat(ignore_index=True)` — no row is filtered,
sorted, deduplicated, or dropped between source read and `to_csv`; `row_in_wave = np.arange(len(df))`
is written at build time, so `(wave, row_in_wave)` indexes source row order directly. No respondent
ID was carried into the pooled file (`S007` exists only in the WV2/WV4/WV5 sources and was never
selected), so a **positional merge is the only option**; it is validated below on three variables
present in both source and pooled file (must be 100% row-for-row, else MERGE INFEASIBLE).

**Weights used per wave:** WV2 `V376` (≡1.0), WV3 `V236`, WV4 `V245`, WV5 `V259`, WV6 `V258`,
WV7 `W_WEIGHT` (≡1.0). **Estimator:** pseudo-maximum-likelihood ordered logit — weights multiply the
per-observation log-likelihood; covariance is the Huber–White sandwich A⁻¹BA⁻¹ (A = −Hessian,
B = Σ weighted per-obs score outer products, scores evaluated numerically). **Caveat:** these SEs
account for weighting only, **not clustering** — WV2–WV6 carry no PSU/stratum identifier (only WV7
has `I_PSU`), so no design-based SE is attempted.

In [2]:
# 1a — merge feasibility gate (positional merge + row-for-row cross-validation)
FILES = {
    "WV2": "F00008272-WV2_Data_Turkey_Excel_v1.6.1.xlsx",
    "WV3": "F00008135-WV3_Data_Turkey_Excel_v20221107.1.xlsx",
    "WV4": "F00008022-WV4_Data_Turkey_Excel_v20201117.1.xlsx",
    "WV5": "F00007873-WV5_Data_Turkey_Excel_v20201117.1.xlsx",
    "WV6": "F00007629-WV6_Data_Turkey_2011_Excel_v20180912.xlsx",
    "WV7": "F00013222-WVS_Wave_7_Turkey_Excel_v5.1.xlsx",
}
WEIGHT_CODE = {"WV2": "V376", "WV3": "V236", "WV4": "V245",
               "WV5": "V259", "WV6": "V258", "WV7": "W_WEIGHT"}
VAL_CODES = {  # crosswalk v3 codes (pool_final.py ROWS): source vars also in the pooled file
    "age":           {"WV2": "V355", "WV3": "V216", "WV4": "V225", "WV5": "V237", "WV6": "V242", "WV7": "Q262"},
    "conf_military": {"WV2": "V273", "WV3": "V136", "WV4": "V148", "WV5": "V132", "WV6": "V109", "WV7": "Q65"},
    "income_raw":    {"WV2": "V363", "WV3": "V227", "WV4": "V236", "WV5": "V253", "WV6": "V239", "WV7": "Q288"},
}

def header_map(path):
    head = pd.read_excel(path, nrows=0)
    return {str(c).split(":", 1)[0].strip(): c for c in head.columns}

pooled_raw = pd.read_csv(f"{RESULTS}/pooled_extended_v2.csv")
wparts, val_report = [], []
for w in WAVES:
    path = FILES[w]
    hm = header_map(path)
    cols = [hm[WEIGHT_CODE[w]]] + [hm[VAL_CODES[v][w]] for v in VAL_CODES]
    src = pd.read_excel(path, usecols=cols).rename(
        columns={hm[WEIGHT_CODE[w]]: "cal_weight",
                 **{hm[VAL_CODES[v][w]]: v for v in VAL_CODES}})
    src["wave"], src["row_in_wave"] = w, np.arange(len(src))
    # replicate pool_final.py transforms on the validation variables
    for v in ("age", "income_raw"):
        x = pd.to_numeric(src[v], errors="coerce")
        src[v] = x.where(x >= 0, np.nan)
    cm = pd.to_numeric(src["conf_military"], errors="coerce")
    cm = cm.where(cm >= 0, np.nan).where(lambda s: s.isin([1, 2, 3, 4]), np.nan)
    src["conf_military"] = 5 - cm
    pw = pooled_raw[pooled_raw["wave"] == w].sort_values("row_in_wave")
    agrees = {}
    for v in VAL_CODES:
        a = pw[v].to_numpy(dtype=float)
        b = src.sort_values("row_in_wave")[v].to_numpy(dtype=float)
        agrees[v] = float(np.mean((a == b) | (np.isnan(a) & np.isnan(b))))
    val_report.append({"wave": w, "src_rows": len(src), "pooled_rows": len(pw),
                       "rows_match": len(src) == len(pw),
                       **{f"agree_{v}": agrees[v] for v in VAL_CODES}})
    wparts.append(src[["wave", "row_in_wave", "cal_weight"]])

vr = pd.DataFrame(val_report)
MERGE_OK = bool(vr["rows_match"].all()
                and all((vr[f"agree_{v}"] == 1.0).all() for v in VAL_CODES))
for _, r in vr.iterrows():
    add_row("1a_merge_validation", "", r["wave"], "merge_validation", np.nan, np.nan,
            int(r["src_rows"]),
            f"rows_match={r['rows_match']}; agree age={r['agree_age']:.4f} "
            f"conf_military={r['agree_conf_military']:.4f} income_raw={r['agree_income_raw']:.4f}")
print(vr.to_string(index=False))
if MERGE_OK:
    dfw = df.merge(pd.concat(wparts, ignore_index=True),
                   on=["wave", "row_in_wave"], how="left")
    assert dfw["cal_weight"].notna().all()
    print("\nMERGE VALID — positional merge on (wave, row_in_wave); 100% agreement on all 3 variables x 6 waves")
else:
    print("\nMERGE INFEASIBLE — check 1 aborted (agreement rates above); proceeding to check 2")


wave  src_rows  pooled_rows  rows_match  agree_age  agree_conf_military  agree_income_raw
 WV2      1030         1030        True        1.0                  1.0               1.0
 WV3      1907         1907        True        1.0                  1.0               1.0
 WV4      3401         3401        True        1.0                  1.0               1.0
 WV5      1346         1346        True        1.0                  1.0               1.0
 WV6      1605         1605        True        1.0                  1.0               1.0
 WV7      2415         2415        True        1.0                  1.0               1.0

MERGE VALID — positional merge on (wave, row_in_wave); 100% agreement on all 3 variables x 6 waves


In [3]:
# 1b/1c/1d — weighted refits (pseudo-ML + sandwich SEs)
class WeightedOrderedModel(OrderedModel):
    """Ordered logit with weights multiplying the per-observation log-likelihood."""
    def __init__(self, endog, exog, weights=None, **kw):
        super().__init__(endog, exog, **kw)
        self._w = np.asarray(weights, dtype=float)
    def loglikeobs(self, params):
        return self._w * super().loglikeobs(params)


def fit_weighted(data, outcome, predictors, start=None):
    sub = data[[outcome] + predictors + ["cal_weight"]].dropna(
        subset=[outcome] + predictors).copy()
    sub[outcome] = sub[outcome].astype(int)
    keep = [p for p in predictors if sub[p].nunique() > 1]
    mod = WeightedOrderedModel(sub[outcome], sub[keep],
                               weights=sub["cal_weight"].to_numpy(), distr="logit")
    res = mod.fit(method="bfgs", maxiter=400, disp=False, start_params=start)
    assert res.mle_retvals.get("converged", False), f"weighted non-convergence: {outcome}"
    p = np.asarray(res.params, dtype=float)
    S = approx_fprime(p, mod.loglikeobs)                      # (n, k) per-obs scores
    H = approx_hess(p, lambda q: mod.loglikeobs(q).sum())
    Ainv = np.linalg.inv(-H)
    cov = Ainv @ (S.T @ S) @ Ainv                             # Huber-White sandwich
    se = pd.Series(np.sqrt(np.diag(cov)), index=res.params.index)
    return res, se, len(sub)


if MERGE_OK:
    # 1b: weight distribution in the army analytic sample
    wsub = dfw[["conf_military"] + PREDICTORS + ["cal_weight", "wave"]].dropna(
        subset=["conf_military"] + PREDICTORS)
    dist = wsub.groupby("wave")["cal_weight"].agg(["count", "min", "max", "mean", "std"])
    dist["cv"] = dist["std"] / dist["mean"]
    print("weight distribution, army analytic sample (n=%d):" % len(wsub))
    print(dist.round(4).to_string())
    for w_, r_ in dist.iterrows():
        add_row("1b_weight_dist", "conf_military", w_, "cal_weight",
                float(r_["mean"]), np.nan, int(r_["count"]),
                f"min={r_['min']:.4f} max={r_['max']:.4f} cv={r_['cv']:.4f}; "
                f"weight var: {WEIGHT_CODE[w_]}")
    # 1c/1d: weighted vs unweighted, army + police + courts
    for inst in ["conf_military", "conf_police", "conf_courts"]:
        unw = res_base if inst == "conf_military" else fit_ord(df, inst, PREDICTORS)[0]
        rw, sew, nw = fit_weighted(dfw, inst, PREDICTORS, start=unw.params.values)
        tab = []
        for t in NR_TERMS:
            if t not in rw.params.index:
                continue
            wc, ws = float(rw.params[t]), float(sew[t])
            wp = 2 * (1 - stats.norm.cdf(abs(wc / ws)))
            tab.append({"term": t,
                        "unweighted": f"{float(unw.params[t]):+.4f} ({float(unw.bse[t]):.4f})",
                        "weighted": f"{wc:+.4f} ({ws:.4f})", "weighted_p": round(wp, 5)})
            add_row("1c_weighted", inst, wave_of(t), t, wc, ws, nw,
                    "pseudo-ML, per-wave calibration weight; sandwich (Huber-White) SE; "
                    "no clustering adjustment (no PSU ids in WV2-WV6)")
        print(f"\n{inst}  weighted n={nw} (unweighted SEs = MLE Hessian, as published):")
        print(pd.DataFrame(tab).to_string(index=False))
        if inst == "conf_military":
            wc, ws = float(rw.params["nr_x_WV4"]), float(sew["nr_x_WV4"])
            SUMMARY["1_weighted"] = ("check 1: weighted (calibration weights, sandwich SE)",
                                     wc, ws, 2 * (1 - stats.norm.cdf(abs(wc / ws))), nw,
                                     f"vs +{g_c:.3f} unweighted; ratio {wc/g_c:.3f}")
else:
    print("skipped: MERGE INFEASIBLE")


weight distribution, army analytic sample (n=8915):
      count     min     max    mean     std      cv
wave                                               
WV2     759  1.0000  1.0000  1.0000  0.0000  0.0000
WV3    1266  0.3430  5.3581  1.0817  0.9107  0.8419
WV4    2669  0.3163  5.5361  1.0208  0.7778  0.7620
WV5    1064  0.7014  1.7313  0.9965  0.2459  0.2468
WV6    1327  0.6560  2.1132  0.9945  0.3533  0.3553
WV7    1830  1.0000  1.0000  1.0000  0.0000  0.0000



conf_military  weighted n=8915 (unweighted SEs = MLE Hessian, as published):
               term       unweighted         weighted  weighted_p
not_religious_dummy -0.7363 (0.1568) -0.7070 (0.1348)     0.00000
           nr_x_WV3 +0.3746 (0.1997) +0.5074 (0.1979)     0.01034
           nr_x_WV4 +0.7162 (0.1828) +0.7760 (0.1833)     0.00002
           nr_x_WV5 +0.6352 (0.2235) +0.5835 (0.2262)     0.00988
           nr_x_WV6 +0.4023 (0.2061) +0.4393 (0.1942)     0.02366
           nr_x_WV7 +0.6511 (0.1819) +0.6383 (0.1565)     0.00005



conf_police  weighted n=8911 (unweighted SEs = MLE Hessian, as published):
               term       unweighted         weighted  weighted_p
not_religious_dummy -0.6918 (0.1521) -0.6837 (0.1427)     0.00000
           nr_x_WV3 +0.2825 (0.1905) +0.3359 (0.2081)     0.10639
           nr_x_WV4 +0.4484 (0.1756) +0.4737 (0.1849)     0.01040
           nr_x_WV5 +0.2609 (0.2062) +0.2793 (0.1908)     0.14337
           nr_x_WV6 +0.1552 (0.2008) +0.1658 (0.1938)     0.39237
           nr_x_WV7 +0.5617 (0.1774) +0.5470 (0.1630)     0.00079



conf_courts  weighted n=6231 (unweighted SEs = MLE Hessian, as published):
               term       unweighted         weighted  weighted_p
not_religious_dummy -0.5030 (0.1549) -0.4964 (0.1575)     0.00162
           nr_x_WV3 +0.2669 (0.1917) +0.2112 (0.2199)     0.33679
           nr_x_WV5 +0.0875 (0.2101) +0.0910 (0.2109)     0.66624
           nr_x_WV6 +0.0549 (0.2029) +0.0271 (0.2103)     0.89736
           nr_x_WV7 +0.0759 (0.1776) +0.0703 (0.1720)     0.68266


## Check 2 — placebo institutions, full statistics + export

**2a** full-sample refits (each institution's own listwise sample, exactly the layer4b sample
logic). **2b** the n = 8,777 common sample per `final_checks.ipynb` cell 15 — non-missing on
`conf_military + conf_parliament + conf_government` + all predictors; police/courts are fitted on
the subset of those rows where their own outcome is observed (n reported; courts loses all WV4 —
item not fielded 2001, so its WV4 terms drop). Totals = main + interaction with **delta-method SE**
from the model covariance, matching `layer4b_longwindow.py::wave_total()`. **2c** both bases are
exported to `results/placebo_models_full.csv` with a `sample_basis` column. **2d** anchor
comparison uses the same 3-dp formatting that produced the stored values.

In [4]:
# 2a — full-sample placebo refits
PLACEBOS = ["conf_parliament", "conf_government", "conf_civil_service"]
PLACEBO_ROWS, full_fits, common_fits = [], {}, {}

def collect(basis, inst, res, n, note=""):
    for t in NR_TERMS:
        if t not in res.params.index:
            continue
        c, s = float(res.params[t]), float(res.bse[t])
        PLACEBO_ROWS.append({"sample_basis": basis, "institution": inst,
            "term_type": "main" if t == "not_religious_dummy" else "interaction",
            "wave": wave_of(t), "term": t, "coef": c, "se": s, "z": c / s,
            "p": 2 * (1 - stats.norm.cdf(abs(c / s))), "n": n, "notes": note})
    for w in WAVES:
        c, s, p = wave_total(res, w)
        if np.isnan(c):
            continue
        PLACEBO_ROWS.append({"sample_basis": basis, "institution": inst,
            "term_type": "total", "wave": w, "term": f"nr_total_{w}",
            "coef": c, "se": s, "z": c / s, "p": p, "n": n,
            "notes": (note + "; " if note else "") + "delta-method SE (layer4b wave_total)"})

for inst in PLACEBOS:
    r, s, _ = fit_ord(df, inst, PREDICTORS)
    full_fits[inst] = r
    collect("full", inst, r, len(s))
    rows = [{"term": t, "coef": round(float(r.params[t]), 4),
             "se": round(float(r.bse[t]), 4), "p": float(f"{float(r.pvalues[t]):.3g}")}
            for t in NR_TERMS]
    rows += [{"term": f"nr_total_{w}", "coef": round(c, 4), "se": round(se_, 4),
              "p": float(f"{p_:.3g}")}
             for w in WAVES for c, se_, p_ in [wave_total(r, w)] if not np.isnan(c)]
    print(f"FULL sample — {inst}  n={len(s)}")
    print(pd.DataFrame(rows).to_string(index=False), "\n")


FULL sample — conf_parliament  n=8873
               term    coef     se            p
not_religious_dummy -0.8911 0.1555 1.000000e-08
           nr_x_WV3  0.6315 0.1943 1.150000e-03
           nr_x_WV4  0.8415 0.1785 2.430000e-06
           nr_x_WV5  0.4101 0.2082 4.880000e-02
           nr_x_WV6  0.4923 0.2008 1.420000e-02
           nr_x_WV7  0.8100 0.1774 4.960000e-06
       nr_total_WV2 -0.8911 0.1555 1.000000e-08
       nr_total_WV3 -0.2595 0.1220 3.330000e-02
       nr_total_WV4 -0.0496 0.0940 5.980000e-01
       nr_total_WV5 -0.4810 0.1428 7.570000e-04
       nr_total_WV6 -0.3987 0.1310 2.330000e-03
       nr_total_WV7 -0.0811 0.0889 3.620000e-01 



FULL sample — conf_government  n=8861
               term    coef     se            p
not_religious_dummy -0.4255 0.1566 6.580000e-03
           nr_x_WV3  0.0641 0.1956 7.430000e-01
           nr_x_WV4  0.3733 0.1804 3.850000e-02
           nr_x_WV5 -0.1603 0.2097 4.450000e-01
           nr_x_WV6 -0.0159 0.2019 9.370000e-01
           nr_x_WV7 -0.1260 0.1788 4.810000e-01
       nr_total_WV2 -0.4255 0.1566 6.580000e-03
       nr_total_WV3 -0.3613 0.1226 3.200000e-03
       nr_total_WV4 -0.0521 0.0960 5.870000e-01
       nr_total_WV5 -0.5857 0.1440 4.750000e-05
       nr_total_WV6 -0.4413 0.1314 7.820000e-04
       nr_total_WV7 -0.5515 0.0901 9.310000e-10 



FULL sample — conf_civil_service  n=8863
               term    coef     se            p
not_religious_dummy -0.7610 0.1515 5.060000e-07
           nr_x_WV3  0.5400 0.1902 4.510000e-03
           nr_x_WV4  0.6454 0.1748 2.230000e-04
           nr_x_WV5  0.4606 0.2067 2.580000e-02
           nr_x_WV6  0.2196 0.1997 2.710000e-01
           nr_x_WV7  0.3344 0.1758 5.710000e-02
       nr_total_WV2 -0.7610 0.1515 5.060000e-07
       nr_total_WV3 -0.2209 0.1215 6.890000e-02
       nr_total_WV4 -0.1156 0.0946 2.220000e-01
       nr_total_WV5 -0.3003 0.1450 3.830000e-02
       nr_total_WV6 -0.5414 0.1341 5.440000e-05
       nr_total_WV7 -0.4265 0.0933 4.840000e-06 



In [5]:
# 2b common sample; 2c export; 2d anchors
TRIO = ["conf_military", "conf_parliament", "conf_government"]
common = df.dropna(subset=TRIO + PREDICTORS).copy()
assert len(common) == 8777, f"common sample n={len(common)} != 8777"
print(f"common sample n = {len(common)} (final_checks cell 15 definition)\n")
ALL6 = ["conf_military", "conf_police", "conf_courts"] + PLACEBOS
for inst in ALL6:
    r, s, _ = fit_ord(common, inst, PREDICTORS)
    common_fits[inst] = r
    note = "cell-15 common sample"
    if len(s) < len(common):
        note += f"; outcome missing on {len(common) - len(s)} common rows"
    if inst == "conf_courts":
        note += "; all WV4 rows drop (item not fielded 2001) -> WV4 terms not in model"
    collect("common_8777", inst, r, len(s), note)
    key = [t for t in ["nr_x_WV4", "nr_x_WV5"] if t in r.params.index]
    print(f"COMMON — {inst:<22} n={len(s):>5}  " + "  ".join(
        f"{t}={float(r.params[t]):+.4f} ({float(r.bse[t]):.4f})" for t in key))
    if inst == "conf_military":
        c_, s_ = float(r.params["nr_x_WV4"]), float(r.bse["nr_x_WV4"])
        SUMMARY["2_common"] = ("check 2: army refit on common_8777 sample", c_, s_,
                               2 * (1 - stats.norm.cdf(abs(c_ / s_))), len(s), "")

placebo_df = pd.DataFrame(PLACEBO_ROWS, columns=["sample_basis", "institution", "term_type",
                                                 "wave", "term", "coef", "se", "z", "p", "n", "notes"])
placebo_df.to_csv(f"{RESULTS}/placebo_models_full.csv", index=False)
print(f"\nwrote results/placebo_models_full.csv ({len(placebo_df)} rows)")
for r in PLACEBO_ROWS:  # mirror into the master table
    ROWS.append({"check": f"2_placebo_{r['sample_basis']}", "institution": r["institution"],
                 "wave": r["wave"], "term": r["term"], "coef": r["coef"], "se": r["se"],
                 "z": r["z"], "p": r["p"], "n": r["n"], "notes": r["notes"]})

# 2d — anchors (compare with the same "+.3f" formatting that produced the stored values)
ANCHORS = [("full", "conf_parliament", "nr_x_WV4", 0.841),
           ("full", "conf_civil_service", "nr_x_WV4", 0.645),
           ("common", "conf_military", "nr_x_WV5", 0.675),
           ("common", "conf_parliament", "nr_x_WV5", 0.440),
           ("common", "conf_government", "nr_x_WV5", -0.167)]
out = []
for basis, inst, term, stored in ANCHORS:
    got = float((full_fits[inst] if basis == "full" else common_fits[inst]).params[term])
    ok = f"{got:+.3f}" == f"{stored:+.3f}"
    out.append({"basis": basis, "institution": inst, "term": term,
                "stored": f"{stored:+.3f}", "refit": f"{got:+.4f}",
                "verdict": "MATCH" if ok else "MISMATCH"})
    add_row("2d_anchor", inst, term.replace("nr_x_", ""), term, got, np.nan, np.nan,
            f"{basis} basis; stored {stored:+.3f} -> {'MATCH' if ok else 'MISMATCH'}")
print(pd.DataFrame(out).to_string(index=False))

# bonus cross-check: every stored 3-dp placebo value (final_checks cell 4 grid + cell 5 totals
# via deck_fig_main.csv)
STORED_INTER = {
    "conf_parliament":    {"WV3": 0.632, "WV4": 0.841, "WV5": 0.410, "WV6": 0.492, "WV7": 0.810},
    "conf_government":    {"WV3": 0.064, "WV4": 0.373, "WV5": -0.160, "WV6": -0.016, "WV7": -0.126},
    "conf_civil_service": {"WV3": 0.540, "WV4": 0.645, "WV5": 0.461, "WV6": 0.220, "WV7": 0.334}}
bad = [(i, w, v, round(float(full_fits[i].params[f"nr_x_{w}"]), 4))
       for i, d in STORED_INTER.items() for w, v in d.items()
       if f"{float(full_fits[i].params[f'nr_x_{w}']):+.3f}" != f"{v:+.3f}"]
fig = pd.read_csv(f"{RESULTS}/deck_fig_main.csv")
NAME = {"parliament": "conf_parliament", "government": "conf_government",
        "civil_service": "conf_civil_service"}
tbad = []
for _, rr in fig[fig["institution"].isin(NAME)].iterrows():
    c, s, p = wave_total(full_fits[NAME[rr["institution"]]], rr["wave"])
    if f"{c:+.3f}" != f"{float(rr['total_effect']):+.3f}":
        tbad.append((rr["institution"], rr["wave"], rr["total_effect"], round(c, 4)))
print(f"\nstored interaction grid (final_checks cell 4): {15 - len(bad)}/15 match; mismatches: {bad or 'none'}")
print(f"stored totals (deck_fig_main.csv <- cell 5):    {18 - len(tbad)}/18 match; mismatches: {tbad or 'none'}")


common sample n = 8777 (final_checks cell 15 definition)



COMMON — conf_military          n= 8777  nr_x_WV4=+0.7379 (0.1871)  nr_x_WV5=+0.6745 (0.2276)


COMMON — conf_police            n= 8748  nr_x_WV4=+0.5199 (0.1799)  nr_x_WV5=+0.3116 (0.2101)


COMMON — conf_courts            n= 6110  nr_x_WV5=+0.0888 (0.2141)


COMMON — conf_parliament        n= 8777  nr_x_WV4=+0.8733 (0.1820)  nr_x_WV5=+0.4398 (0.2110)


COMMON — conf_government        n= 8777  nr_x_WV4=+0.3761 (0.1813)  nr_x_WV5=-0.1667 (0.2103)


COMMON — conf_civil_service     n= 8711  nr_x_WV4=+0.6619 (0.1793)  nr_x_WV5=+0.4892 (0.2108)

wrote results/placebo_models_full.csv (106 rows)
 basis        institution     term stored   refit verdict
  full    conf_parliament nr_x_WV4 +0.841 +0.8415   MATCH
  full conf_civil_service nr_x_WV4 +0.645 +0.6454   MATCH
common      conf_military nr_x_WV5 +0.675 +0.6745   MATCH
common    conf_parliament nr_x_WV5 +0.440 +0.4398   MATCH
common    conf_government nr_x_WV5 -0.167 -0.1667   MATCH

stored interaction grid (final_checks cell 4): 15/15 match; mismatches: none
stored totals (deck_fig_main.csv <- cell 5):    18/18 match; mismatches: none


## Check 3 — rel_attendance as control

`rel_attendance` (1 = more than once a week … 8 = never) is a control in the layer4b spec while
`not_religious_dummy` is the grouping variable — conditioning on a component of the construct.
**3a/3c** drop it and re-apply the same listwise sample logic (n grows: its missing rows return).
**3b** descriptive only: mean attendance by nr group by wave, full pooled file.

In [6]:
# 3a/3c — refits without rel_attendance; 3b — descriptive
PRED_NO_ATT = [p for p in PREDICTORS if p != "rel_attendance"]
for inst in ["conf_military", "conf_police", "conf_courts"]:
    r, s, _ = fit_ord(df, inst, PRED_NO_ATT)
    tab = []
    for t in NR_TERMS:
        if t not in r.params.index:
            continue
        b = pub.query("institution==@inst and predictor==@t")
        tab.append({"term": t,
                    "baseline(published)": f"{float(b['coef'].iloc[0]):+.4f} ({float(b['se'].iloc[0]):.4f})",
                    "no_rel_attendance": f"{float(r.params[t]):+.4f} ({float(r.bse[t]):.4f})",
                    "p": float(f"{float(r.pvalues[t]):.3g}")})
        add_row("3a_no_rel_attendance", inst, wave_of(t), t,
                float(r.params[t]), float(r.bse[t]), len(s),
                "rel_attendance dropped from controls; layer4b sample logic re-applied")
    print(f"{inst}: baseline n={int(pub.query('institution==@inst')['n_obs'].iloc[0])} -> no-attendance n={len(s)}")
    print(pd.DataFrame(tab).to_string(index=False), "\n")
    if inst == "conf_military":
        SUMMARY["3_no_att"] = ("check 3: rel_attendance dropped from controls",
                               float(r.params["nr_x_WV4"]), float(r.bse["nr_x_WV4"]),
                               float(r.pvalues["nr_x_WV4"]), len(s), "")

# 3b — mean rel_attendance by nr x wave (full pooled file; higher = LESS frequent attendance)
raw3 = pd.read_csv(f"{RESULTS}/pooled_extended_v2.csv")
raw3["not_religious_dummy"] = raw3["rel_person"].map(
    lambda v: 1 if v in (2, 3) else (0 if v == 1 else np.nan))
d3 = raw3.dropna(subset=["rel_attendance", "not_religious_dummy"])
g = d3.groupby(["wave", "not_religious_dummy"])["rel_attendance"].agg(["mean", "count"])
t3 = g["mean"].unstack().rename(columns={0.0: "religious", 1.0: "not_religious"})
t3["gap"] = t3["not_religious"] - t3["religious"]
print("mean rel_attendance (1=+weekly .. 8=never), full pooled file:")
print(t3.round(3).to_string())
for (w_, nr_), r_ in g.iterrows():
    add_row("3b_descriptive", "", w_, f"mean_rel_attendance_nr{int(nr_)}",
            float(r_["mean"]), np.nan, int(r_["count"]),
            "full pooled file; scale 1=+weekly..8=never")


conf_military: baseline n=8915 -> no-attendance n=9116
               term baseline(published) no_rel_attendance        p
not_religious_dummy    -0.7363 (0.1568)  -0.7551 (0.1557) 0.000001
           nr_x_WV3    +0.3746 (0.1997)  +0.3900 (0.1985) 0.049400
           nr_x_WV4    +0.7162 (0.1828)  +0.7261 (0.1814) 0.000063
           nr_x_WV5    +0.6352 (0.2235)  +0.5647 (0.2198) 0.010200
           nr_x_WV6    +0.4023 (0.2061)  +0.4072 (0.2053) 0.047400
           nr_x_WV7    +0.6511 (0.1819)  +0.5867 (0.1796) 0.001090 



conf_police: baseline n=8911 -> no-attendance n=9110
               term baseline(published) no_rel_attendance            p
not_religious_dummy    -0.6918 (0.1521)  -0.7654 (0.1512) 4.140000e-07
           nr_x_WV3    +0.2825 (0.1905)  +0.3124 (0.1895) 9.920000e-02
           nr_x_WV4    +0.4484 (0.1756)  +0.4808 (0.1745) 5.850000e-03
           nr_x_WV5    +0.2609 (0.2062)  +0.2315 (0.2039) 2.560000e-01
           nr_x_WV6    +0.1552 (0.2008)  +0.1737 (0.2002) 3.850000e-01
           nr_x_WV7    +0.5617 (0.1774)  +0.5700 (0.1751) 1.130000e-03 



conf_courts: baseline n=6231 -> no-attendance n=6390
               term baseline(published) no_rel_attendance        p
not_religious_dummy    -0.5030 (0.1549)  -0.5811 (0.1532) 0.000149
           nr_x_WV3    +0.2669 (0.1917)  +0.3047 (0.1905) 0.110000
           nr_x_WV5    +0.0875 (0.2101)  +0.0965 (0.2070) 0.641000
           nr_x_WV6    +0.0549 (0.2029)  +0.0911 (0.2019) 0.652000
           nr_x_WV7    +0.0759 (0.1776)  +0.0978 (0.1751) 0.577000 

mean rel_attendance (1=+weekly .. 8=never), full pooled file:
not_religious_dummy  religious  not_religious    gap
wave                                                
WV2                      4.108          5.516  1.407
WV3                      3.622          5.154  1.532
WV4                      4.116          5.129  1.013
WV5                      3.992          5.373  1.382
WV6                      3.963          5.186  1.223
WV7                      3.385          4.593  1.207


## Check 4 — left_right as control

`left_right` is the largest single attrition source (737 rows missing it alone). **4a** drop it and
re-apply the sample logic (larger n). **4b** decomposition per `education_defense.ipynb`: also fit
the no-LR spec on the original 8,915 rows, so movement splits into control-removal vs added
respondents.

In [7]:
# 4a/4b — drop left_right, with sample-composition decomposition
PRED_NO_LR = [p for p in PREDICTORS if p != "left_right"]
r_enl, s_enl, _ = fit_ord(df, "conf_military", PRED_NO_LR)
r_same, s_same, _ = fit_ord(df.loc[sub_base.index], "conf_military", PRED_NO_LR)
assert len(s_same) == 8915
tab = []
for t in NR_TERMS:
    tab.append({"term": t,
                "baseline_8915_withLR": f"{float(res_base.params[t]):+.4f} ({float(res_base.bse[t]):.4f})",
                "noLR_same_8915": f"{float(r_same.params[t]):+.4f} ({float(r_same.bse[t]):.4f})",
                f"noLR_enlarged_{len(s_enl)}": f"{float(r_enl.params[t]):+.4f} ({float(r_enl.bse[t]):.4f})"})
    add_row("4b_no_left_right_same8915", "conf_military", wave_of(t), t,
            float(r_same.params[t]), float(r_same.bse[t]), len(s_same),
            "left_right dropped; original layer4b sample held fixed (spec effect)")
    add_row("4a_no_left_right_enlarged", "conf_military", wave_of(t), t,
            float(r_enl.params[t]), float(r_enl.bse[t]), len(s_enl),
            "left_right dropped; sample logic re-applied (spec + composition)")
print(f"enlarged n = {len(s_enl)} (recovers {len(s_enl) - 8915} respondents)")
print(pd.DataFrame(tab).to_string(index=False))
d_spec = float(r_same.params["nr_x_WV4"]) - g_c
d_comp = float(r_enl.params["nr_x_WV4"]) - float(r_same.params["nr_x_WV4"])
print(f"\nnr_x_WV4 decomposition: control removal {d_spec:+.4f}, added respondents {d_comp:+.4f}")
SUMMARY["4_no_lr"] = ("check 4: left_right dropped (enlarged sample)",
                      float(r_enl.params["nr_x_WV4"]), float(r_enl.bse["nr_x_WV4"]),
                      float(r_enl.pvalues["nr_x_WV4"]), len(s_enl),
                      f"same-8915 variant: {float(r_same.params['nr_x_WV4']):+.4f} "
                      f"({float(r_same.bse['nr_x_WV4']):.4f})")


enlarged n = 9652 (recovers 737 respondents)
               term baseline_8915_withLR   noLR_same_8915 noLR_enlarged_9652
not_religious_dummy     -0.7363 (0.1568) -0.7651 (0.1566)   -0.7624 (0.1501)
           nr_x_WV3     +0.3746 (0.1997) +0.3552 (0.1993)   +0.3566 (0.1929)
           nr_x_WV4     +0.7162 (0.1828) +0.6571 (0.1824)   +0.6807 (0.1764)
           nr_x_WV5     +0.6352 (0.2235) +0.5751 (0.2228)   +0.5617 (0.2125)
           nr_x_WV6     +0.4023 (0.2061) +0.3508 (0.2057)   +0.3436 (0.1988)
           nr_x_WV7     +0.6511 (0.1819) +0.6482 (0.1819)   +0.6555 (0.1749)

nr_x_WV4 decomposition: control removal -0.0591, added respondents +0.0236


## Check 5 — link function / proportional odds

No Brant test in statsmodels → **informal Brant**: separate binary logits at the three cutpoints
(y ≥ 2, y ≥ 3, y ≥ 4) with the identical spec; under PO the slope coefficients are equal across
cutpoints. Pairwise contrasts use the independence SE — an **upper bound** (same respondents ⇒
positive cross-fit covariance), so significant differences are robust, non-significant ones
inconclusive. **5b** the y ≥ 3 logit ("a great deal" + "quite a lot" vs not) is the claim's own
cutpoint — the binary replication that has to hold. **5c** same for police/courts.

In [8]:
# 5a — informal Brant (cutpoint logits, army sample); 5b/5c — binary replication
cutfits = {}
for cut in (2, 3, 4):
    y = (sub_base["conf_military"] >= cut).astype(int)
    rb = sm.Logit(y, sm.add_constant(sub_base[keep_base])).fit(disp=False, maxiter=200)
    assert rb.mle_retvals["converged"]
    cutfits[cut] = rb
    print(f"cutpoint y>={cut}: P(y>={cut}) = {y.mean():.3f}")
    for t in NR_TERMS:
        add_row("5a_cutpoint_logit", "conf_military", wave_of(t), t,
                float(rb.params[t]), float(rb.bse[t]), len(y),
                f"binary logit at cutpoint y>={cut} (informal Brant)")
tab = [{"term": t,
        **{f"y>={c}": f"{float(cutfits[c].params[t]):+.4f} ({float(cutfits[c].bse[t]):.4f})" for c in (2, 3, 4)},
        "ordinal": f"{float(res_base.params[t]):+.4f} ({float(res_base.bse[t]):.4f})"}
       for t in NR_TERMS]
print(); print(pd.DataFrame(tab).to_string(index=False))

print("\npairwise cutpoint contrasts (conservative independence SE):")
po_flags = []
for t in ["not_religious_dummy", "nr_x_WV4"]:
    for c1, c2 in [(2, 3), (2, 4), (3, 4)]:
        b1, s1 = float(cutfits[c1].params[t]), float(cutfits[c1].bse[t])
        b2, s2 = float(cutfits[c2].params[t]), float(cutfits[c2].bse[t])
        z = (b1 - b2) / np.sqrt(s1**2 + s2**2)
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        po_flags.append(p < 0.05)
        print(f"  {t:<22} y>={c1} vs y>={c2}: diff {b1-b2:+.4f}  z {z:+.2f}  p {p:.3f}")
        add_row("5a_po_contrast", "conf_military", "", f"{t}_y{c1}_vs_y{c2}",
                b1 - b2, float(np.sqrt(s1**2 + s2**2)), len(sub_base),
                "conservative independence SE (upper bound; same-sample fits)")
print("PO verdict on nr / nr_x_WV4:", "VIOLATED" if any(po_flags) else
      "not rejected (all contrasts n.s. under the conservative test)")

# 5b/5c — binary y>=3 replication, all three institutions (own layer4b samples)
for inst in ["conf_military", "conf_police", "conf_courts"]:
    if inst == "conf_military":
        r_ord, s_, k_ = res_base, sub_base, keep_base
        rb = cutfits[3]
    else:
        r_ord, s_, k_ = fit_ord(df, inst, PREDICTORS)
        yb = (s_[inst] >= 3).astype(int)
        rb = sm.Logit(yb, sm.add_constant(s_[k_])).fit(disp=False, maxiter=200)
        assert rb.mle_retvals["converged"]
    tab = [{"term": t, "ordinal": f"{float(r_ord.params[t]):+.4f}",
            "binary_y>=3": f"{float(rb.params[t]):+.4f} ({float(rb.bse[t]):.4f})",
            "binary_p": float(f"{float(rb.pvalues[t]):.3g}")}
           for t in NR_TERMS if t in rb.params.index]
    print(f"\n{inst}  n={len(s_)}  (binary = P(high confidence): categories 3-4)")
    print(pd.DataFrame(tab).to_string(index=False))
    for t in NR_TERMS:
        if t in rb.params.index:
            add_row("5b_binary_high", inst, wave_of(t), t,
                    float(rb.params[t]), float(rb.bse[t]), len(s_),
                    "binary logit, high confidence (y>=3) vs not; identical spec + constant")
SUMMARY["5_binary"] = ("check 5: binary logit, high confidence (y>=3)",
                       float(cutfits[3].params["nr_x_WV4"]), float(cutfits[3].bse["nr_x_WV4"]),
                       float(cutfits[3].pvalues["nr_x_WV4"]), int(len(sub_base)),
                       "single-cutpoint logit scale; compare sign/significance, not magnitude")


cutpoint y>=2: P(y>=2) = 0.948


cutpoint y>=3: P(y>=3) = 0.864


cutpoint y>=4: P(y>=4) = 0.575

               term             y>=2             y>=3             y>=4          ordinal
not_religious_dummy -1.6365 (0.6649) -1.2266 (0.2945) -0.8847 (0.1797) -0.7363 (0.1568)
           nr_x_WV3 +1.3247 (0.7655) +0.7938 (0.3686) +0.4414 (0.2245) +0.3746 (0.1997)
           nr_x_WV4 +1.6344 (0.6824) +1.3228 (0.3220) +0.8831 (0.2051) +0.7162 (0.1828)
           nr_x_WV5 +1.2851 (0.7220) +1.0222 (0.3544) +0.9312 (0.2431) +0.6352 (0.2235)
           nr_x_WV6 +1.6395 (0.6982) +0.9744 (0.3333) +0.5731 (0.2401) +0.4023 (0.2061)
           nr_x_WV7 +1.0378 (0.7255) +1.2075 (0.3285) +0.7262 (0.2081) +0.6511 (0.1819)

pairwise cutpoint contrasts (conservative independence SE):
  not_religious_dummy    y>=2 vs y>=3: diff -0.4099  z -0.56  p 0.573
  not_religious_dummy    y>=2 vs y>=4: diff -0.7518  z -1.09  p 0.275
  not_religious_dummy    y>=3 vs y>=4: diff -0.3419  z -0.99  p 0.322
  nr_x_WV4               y>=2 vs y>=3: diff +0.3116  z +0.41  p 0.680
  nr_x_WV4 


conf_police  n=8911  (binary = P(high confidence): categories 3-4)
               term ordinal      binary_y>=3  binary_p
not_religious_dummy -0.6918 -0.5887 (0.1827)   0.00128
           nr_x_WV3 +0.2825 +0.1581 (0.2317)   0.49500
           nr_x_WV4 +0.4484 +0.4244 (0.2102)   0.04350
           nr_x_WV5 +0.2609 +0.0754 (0.2509)   0.76400
           nr_x_WV6 +0.1552 +0.0175 (0.2422)   0.94200
           nr_x_WV7 +0.5617 +0.4365 (0.2292)   0.05690



conf_courts  n=6231  (binary = P(high confidence): categories 3-4)
               term ordinal      binary_y>=3  binary_p
not_religious_dummy -0.5030 -0.4537 (0.1766)    0.0102
           nr_x_WV3 +0.2669 +0.3666 (0.2212)    0.0975
           nr_x_WV5 +0.0875 +0.0480 (0.2426)    0.8430
           nr_x_WV6 +0.0549 -0.0546 (0.2316)    0.8140
           nr_x_WV7 +0.0759 +0.1444 (0.2100)    0.4920


## Summary — robustness of the 2001 dating (`nr x WV4`, army) in one place

In [9]:
# master tidy table + one-row-per-check summary
master = pd.DataFrame(ROWS, columns=["check", "institution", "wave", "term",
                                     "coef", "se", "z", "p", "n", "notes"])
master.to_csv(f"{RESULTS}/preflight_refits_table.csv", index=False)
print(f"wrote results/preflight_refits_table.csv ({len(master)} rows)")
print(f"wrote results/placebo_models_full.csv ({len(placebo_df)} rows)\n")

order = ["0_baseline", "1_weighted", "2_common", "3_no_att", "4_no_lr", "5_binary"]
summ = pd.DataFrame([{"variation": SUMMARY[k][0],
                      "nr_x_WV4": f"{SUMMARY[k][1]:+.4f}",
                      "se": f"{SUMMARY[k][2]:.4f}",
                      "p": f"{SUMMARY[k][3]:.3g}",
                      "n": SUMMARY[k][4],
                      "note": SUMMARY[k][5]}
                     for k in order if k in SUMMARY])
summ


wrote results/preflight_refits_table.csv (228 rows)
wrote results/placebo_models_full.csv (106 rows)



,variation,nr_x_WV4,se,p,n,note
0,layer4b baseline (gate refit = published),+0.7162,0.1828,8.95e-05,8915,
1,"check 1: weighted (calibration weights, sandwi...",+0.7760,0.1833,2.3e-05,8915,vs +0.716 unweighted; ratio 1.084
2,check 2: army refit on common_8777 sample,+0.7379,0.1871,8.03e-05,8777,
3,check 3: rel_attendance dropped from controls,+0.7261,0.1814,6.26e-05,9116,
4,check 4: left_right dropped (enlarged sample),+0.6807,0.1764,0.000114,9652,same-8915 variant: +0.6571 (0.1824)
5,"check 5: binary logit, high confidence (y>=3)",+1.3228,0.3220,3.99e-05,8915,single-cutpoint logit scale; compare sign/sign...
